# I. Chuẩn bị và khám phá dữ liệu ban đầu

## 1. Import thư viện

In [ ]:
# Xử lý dữ liệu cơ bản
import numpy as np  
import pandas as pd  
import re  
from collections import Counter  

# Tiền xử lý dữ liệu
from sklearn.model_selection import train_test_split, GridSearchCV  
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.preprocessing import LabelEncoder

## 2. Tải tập dữ liệu

In [50]:
df = pd.read_excel(r"D:\Năm3_Kì2\CAP1\DuLieu_Final.xlsx")

## 3. Tổng quan bộ dữ liệu

Trước tiên, thực hiện phân tích sơ bộ để hiểu rõ cấu trúc và kiểu dữ liệu của các cột:

In [51]:
df.head()

,ID,Movie_Title,Link,Released_Date,Released_Year,Countries_of_origin,Genre,Runtime,IMDB_Rating,No_of_votes,Director,Stars,Gross,Overview
0,MV1,The Praetorian,https://www.imdb.com/title/tt11463896/,"February 12, 2020 (United States)",2021.0,United States,Thriller,NaN,7.6,14,Nelson Ricardo,Nelson Ricardo | Bello Patricia | Nicholas Apo...,NaN,David Donovan was a Bodyguard for hire who wil...
1,MV2,365 Days,https://www.imdb.com/title/tt10886166/,"June 7, 2020 (United States)",2020.0,Poland,Drama|Romance,1h 54m,3.3,107.9K,Barbara Bialowas | Tomasz Mandes,Anna-Maria Sieklucka | Michele Morrone | Broni...,"$9,458,590",Massimo is a member of the Sicilian Mafia fami...
2,MV3,Promising Young Woman,https://www.imdb.com/title/tt9620292/,"December 25, 2020 (United States)",2020.0,United Kingdom|United States,Crime|Drama|Mystery,1h 53m,7.5,253.1K,Emerald Fennell,Carey Mulligan | Bo Burnham | Alison Brie,"$18,854,166",An unexpected encounter gives a wickedly smart...
3,MV4,The Father,https://www.imdb.com/title/tt10272386/,"February 26, 2021 (United States)",2020.0,United Kingdom|France|United States,Drama|Mystery,1h 37m,8.2,242.2K,Florian Zeller,Anthony Hopkins | Olivia Colman | Mark Gatiss,"$24,048,935",A man refuses all assistance from his daughter...
4,MV5,The Invisible Man,https://www.imdb.com/title/tt1051906/,"February 28, 2020 (United States)",2020.0,Australia|United States,Drama|Horror|Mystery,2h 4m,7.1,281.8K,Leigh Whannell,Elisabeth Moss | Oliver Jackson-Cohen | Harrie...,"$144,492,724",When Cecilia's abusive ex takes his own life a...


In [52]:
print(f'Số hàng của bộ dữ liệu này là {df.shape[0]}, và Số cột là {df.shape[1]}')

Số hàng của bộ dữ liệu này là 63389, và Số cột là 14


In [53]:
df.info()
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63389 entries, 0 to 63388
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   63389 non-null  object 
 1   Movie_Title          63384 non-null  object 
 2   Link                 63389 non-null  object 
 3   Released_Date        62949 non-null  object 
 4   Released_Year        63374 non-null  float64
 5   Countries_of_origin  62278 non-null  object 
 6   Genre                61973 non-null  object 
 7   Runtime              55136 non-null  object 
 8   IMDB_Rating          63389 non-null  float64
 9   No_of_votes          63382 non-null  object 
 10  Director             62464 non-null  object 
 11  Stars                59095 non-null  object 
 12  Gross                14220 non-null  object 
 13  Overview             58163 non-null  object 
dtypes: float64(2), object(12)
memory usage: 6.8+ MB


Index(['ID', 'Movie_Title', 'Link', 'Released_Date', 'Released_Year',
       'Countries_of_origin', 'Genre', 'Runtime', 'IMDB_Rating', 'No_of_votes',
       'Director', 'Stars', 'Gross', 'Overview'],
      dtype='object')

In [54]:
df.duplicated().sum()

np.int64(0)

In [55]:
# Kiểm tra giá trị thiếu
df.isnull().sum()

ID                         0
Movie_Title                5
Link                       0
Released_Date            440
Released_Year             15
Countries_of_origin     1111
Genre                   1416
Runtime                 8253
IMDB_Rating                0
No_of_votes                7
Director                 925
Stars                   4294
Gross                  49169
Overview                5226
dtype: int64

# II. Xử lý dữ liệu

## 1. Xử lý giá trị thiếu

In [56]:
# Xóa dòng thiếu Movie_Title
df = df.dropna(subset=['Movie_Title'])

In [57]:
# Xóa dòng thiếu No_of_votes
df = df.dropna(subset=['No_of_votes'])

In [58]:
# Link không dùng trong phân tích nên bỏ
df = df.drop(columns=['Link'])

## 2. Xử lý Released_Date và Release_Country

In [59]:
# --- 1. Tách country ---
df['Release_Country'] = df['Released_Date'].str.extract(r'\((.*?)\)')

# --- 2. Xóa phần (country) khỏi date ---
df['Released_Date'] = df['Released_Date'].str.replace(r'\s*\(.*?\)', '', regex=True).str.strip()

In [60]:
# Điền Countries_of_origin bằng Release_Country nếu bị thiếu
df['Countries_of_origin'] = df['Countries_of_origin'].fillna(df['Release_Country'])

## 3. Chuẩn hóa Released_Date

In [61]:
# Mask: dòng có full date
mask_full_date = df['Released_Date'].str.contains(',', na=False)

# Chỉ convert những dòng có đầy đủ ngày tháng năm
df.loc[mask_full_date, 'Released_Date'] = pd.to_datetime(
    df.loc[mask_full_date, 'Released_Date'],
    errors='coerce'
)

In [62]:
# Điền những giá trị Released_Date bị trống bằng Released_Year
df['Released_Date'] = df['Released_Date'].fillna(df['Released_Year'].astype('Int64').astype(str))

In [63]:
# Parse datetime tạm để lấy phân bố tháng
df['Released_Date_parsed'] = pd.to_datetime(df['Released_Date'], errors='coerce')

In [64]:
# Xác định dòng chỉ có năm
mask_year_only = df['Released_Date'].str.match(r'^\d{4}$', na=False)

In [65]:
# Parse datetime tạm để lấy distribution
df['Released_Date_parsed'] = pd.to_datetime(df['Released_Date'], errors='coerce')

In [66]:
# Lấy distribution từ những dòng có date đầy đủ
df_known = df[~mask_year_only]

month_dist = (
    df_known['Released_Date_parsed']
    .dt.month
    .value_counts(normalize=True)
    .sort_index()
)

month_dist = month_dist.reindex(range(1, 13), fill_value=0)

In [67]:
# Impute ngày/tháng cho những dòng chỉ có năm
def generate_date(year, month):
    day = np.random.randint(1, 29)
    return pd.Timestamp(year=int(year), month=int(month), day=int(day))

imputed_dates = []

for idx, row in df[mask_year_only].iterrows():
    year = int(row['Released_Date'])
    
    month = np.random.choice(month_dist.index, p=month_dist.values)
    imputed_dates.append(generate_date(year, month))

In [68]:
# Ghi đè lại vào cột Released_Date
df.loc[mask_year_only, 'Released_Date'] = imputed_dates

In [69]:
# Convert toàn bộ Released_Date về datetime
df['Released_Date'] = pd.to_datetime(df['Released_Date'], errors='coerce')

In [70]:
# Gắn cờ cho những dòng bị điền ngày/tháng
df['is_imputed_date'] = mask_year_only.astype(int)

## 4. Lọc phim trong khoảng năm 2020–2025

In [71]:
# Lọc dữ liệu để chỉ giữ phim có năm phát hành từ 2020 đến 2025
df = df[
    df['Released_Date'].dt.year.between(2020, 2025)
].copy()

# Kiểm tra lại khoảng năm sau khi lọc
print("Năm nhỏ nhất:", df['Released_Date'].dt.year.min())
print("Năm lớn nhất:", df['Released_Date'].dt.year.max())
print("Số dòng sau khi lọc 2020-2025:", df.shape[0])

Năm nhỏ nhất: 2020
Năm lớn nhất: 2025
Số dòng sau khi lọc 2020-2025: 61455


## 5. Kiểm tra dữ liệu sau xử lý ngày và lọc năm

In [72]:
df.head()

,ID,Movie_Title,Released_Date,Released_Year,Countries_of_origin,Genre,Runtime,IMDB_Rating,No_of_votes,Director,Stars,Gross,Overview,Release_Country,Released_Date_parsed,is_imputed_date
0,MV1,The Praetorian,2020-02-12,2021.0,United States,Thriller,NaN,7.6,14,Nelson Ricardo,Nelson Ricardo | Bello Patricia | Nicholas Apo...,NaN,David Donovan was a Bodyguard for hire who wil...,United States,2020-02-12,0
1,MV2,365 Days,2020-06-07,2020.0,Poland,Drama|Romance,1h 54m,3.3,107.9K,Barbara Bialowas | Tomasz Mandes,Anna-Maria Sieklucka | Michele Morrone | Broni...,"$9,458,590",Massimo is a member of the Sicilian Mafia fami...,United States,2020-06-07,0
2,MV3,Promising Young Woman,2020-12-25,2020.0,United Kingdom|United States,Crime|Drama|Mystery,1h 53m,7.5,253.1K,Emerald Fennell,Carey Mulligan | Bo Burnham | Alison Brie,"$18,854,166",An unexpected encounter gives a wickedly smart...,United States,2020-12-25,0
3,MV4,The Father,2021-02-26,2020.0,United Kingdom|France|United States,Drama|Mystery,1h 37m,8.2,242.2K,Florian Zeller,Anthony Hopkins | Olivia Colman | Mark Gatiss,"$24,048,935",A man refuses all assistance from his daughter...,United States,2021-02-26,0
4,MV5,The Invisible Man,2020-02-28,2020.0,Australia|United States,Drama|Horror|Mystery,2h 4m,7.1,281.8K,Leigh Whannell,Elisabeth Moss | Oliver Jackson-Cohen | Harrie...,"$144,492,724",When Cecilia's abusive ex takes his own life a...,United States,2020-02-28,0


In [73]:
df.isnull().sum()

ID                          0
Movie_Title                 0
Released_Date               0
Released_Year              14
Countries_of_origin        27
Genre                    1311
Runtime                  7977
IMDB_Rating                 0
No_of_votes                 0
Director                  902
Stars                    4183
Gross                   47700
Overview                 5084
Release_Country           436
Released_Date_parsed        0
is_imputed_date             0
dtype: int64

## 6. Xử lý Genre và xóa cột không cần thiết

In [74]:
# Xóa cột Released_Year và Released_Date_parsed
df = df.drop(columns=['Released_Year', 'Released_Date_parsed'])

# Điền Unknown cho Genre bị thiếu
df['Genre'] = df['Genre'].fillna('Unknown')

# Xóa dòng thiếu Genre nếu vẫn còn
df = df.dropna(subset=['Genre'])

## 7. Chuyển No_of_votes về kiểu số

In [75]:
def convert_votes(x):
    if pd.isna(x):
        return None
    
    x = str(x).replace(',', '').strip()
    
    if 'K' in x:
        return float(x.replace('K', '')) * 1_000
    elif 'M' in x:
        return float(x.replace('M', '')) * 1_000_000
    else:
        return float(x)

df['No_of_votes'] = df['No_of_votes'].apply(convert_votes)

## 8. Tính Weighted_Rating

In [76]:
# Đảm bảo IMDB_Rating và No_of_votes là kiểu số
df['IMDB_Rating'] = pd.to_numeric(df['IMDB_Rating'], errors='coerce')
df['No_of_votes'] = pd.to_numeric(df['No_of_votes'], errors='coerce')

# Trung bình rating toàn bộ tập dữ liệu sau khi đã lọc 2020-2025
C = df['IMDB_Rating'].mean(skipna=True)

# Ngưỡng vote tối thiểu
m = 1000

# Số vote của từng phim
v = df['No_of_votes'].fillna(0)

# Rating gốc của từng phim
R = df['IMDB_Rating']

# Công thức Weighted Rating kiểu IMDb
df['Weighted_Rating'] = ((v / (v + m)) * R) + ((m / (v + m)) * C)

# Xem thử kết quả
df[['Movie_Title', 'IMDB_Rating', 'No_of_votes', 'Weighted_Rating']].head()

,Movie_Title,IMDB_Rating,No_of_votes,Weighted_Rating
0,The Praetorian,7.6,14.0,6.293892
1,365 Days,3.3,107900.0,3.327324
2,Promising Young Woman,7.5,253100.0,7.495181
3,The Father,8.2,242200.0,8.192087
4,The Invisible Man,7.1,281800.0,7.097085


## 9. Xử lý Runtime

In [77]:
def convert_runtime(x):
    if pd.isna(x):
        return None
    
    hours = 0
    minutes = 0
    
    h = re.search(r'(\d+)h', x)
    m = re.search(r'(\d+)m', x)
    
    if h:
        hours = int(h.group(1))
    if m:
        minutes = int(m.group(1))
    
    return hours * 60 + minutes

df['Runtime_min'] = df['Runtime'].apply(convert_runtime)
df = df.drop(columns=['Runtime'])

## 10. Xử lý Gross

In [78]:
df['Gross'] = (
    df['Gross']
    .replace('[\$,]', '', regex=True)
)

df['Gross'] = pd.to_numeric(df['Gross'], errors='coerce')

## 11. Điền giá trị thiếu cho các cột mô tả, đạo diễn, diễn viên

In [79]:
# Điền No description cho Overview bị thiếu
df['Overview'] = df['Overview'].fillna('No description')

In [80]:
# Điền Unknown cho Director và Stars
df['Director'] = df['Director'].fillna('Unknown')
df['Stars'] = df['Stars'].fillna('Unknown')

## 12. Tạo cột has_gross

In [81]:
# has_gross = 1 nếu có doanh thu, = 0 nếu không có
df['has_gross'] = df['Gross'].notna().astype(int)

## 13. Điền Runtime_min theo trung vị của Genre

In [82]:
df['Runtime_min'] = df.groupby('Genre')['Runtime_min'].transform(
    lambda x: x.fillna(x.median())
)

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keep

## 14. Xử lý Release_Country

In [83]:
# Những giá trị Release_Country bị thiếu thì điền bằng quốc gia đầu tiên trong Countries_of_origin
df['Release_Country'] = df['Release_Country'].fillna(
    df['Countries_of_origin'].str.split('|').str[0]
)

## 15. Kiểm tra và xóa các giá trị thiếu còn lại

In [84]:
df.isnull().sum()

ID                         0
Movie_Title                0
Released_Date              0
Countries_of_origin       27
Genre                      0
IMDB_Rating                0
No_of_votes                0
Director                   0
Stars                      0
Gross                  47706
Overview                   0
Release_Country           27
is_imputed_date            0
Weighted_Rating            0
Runtime_min               40
has_gross                  0
dtype: int64

In [85]:
# Xóa nốt những giá trị bị trống trong 3 cột
df = df.dropna(subset=['Runtime_min', 'Release_Country', 'Countries_of_origin'])

In [86]:
df.isnull().sum()

ID                         0
Movie_Title                0
Released_Date              0
Countries_of_origin        0
Genre                      0
IMDB_Rating                0
No_of_votes                0
Director                   0
Stars                      0
Gross                  47642
Overview                   0
Release_Country            0
is_imputed_date            0
Weighted_Rating            0
Runtime_min                0
has_gross                  0
dtype: int64

## 16. Kiểm tra dữ liệu cuối cùng trước khi lưu

In [87]:
print(f'Số hàng của bộ dữ liệu sau khi xử lý là {df.shape[0]}, và Số cột là {df.shape[1]}')

Số hàng của bộ dữ liệu sau khi xử lý là 61388, và Số cột là 16


In [88]:
df.head()

,ID,Movie_Title,Released_Date,Countries_of_origin,Genre,IMDB_Rating,No_of_votes,Director,Stars,Gross,Overview,Release_Country,is_imputed_date,Weighted_Rating,Runtime_min,has_gross
0,MV1,The Praetorian,2020-02-12,United States,Thriller,7.6,14.0,Nelson Ricardo,Nelson Ricardo | Bello Patricia | Nicholas Apo...,NaN,David Donovan was a Bodyguard for hire who wil...,United States,0,6.293892,93.0,0
1,MV2,365 Days,2020-06-07,Poland,Drama|Romance,3.3,107900.0,Barbara Bialowas | Tomasz Mandes,Anna-Maria Sieklucka | Michele Morrone | Broni...,9458590.0,Massimo is a member of the Sicilian Mafia fami...,United States,0,3.327324,114.0,1
2,MV3,Promising Young Woman,2020-12-25,United Kingdom|United States,Crime|Drama|Mystery,7.5,253100.0,Emerald Fennell,Carey Mulligan | Bo Burnham | Alison Brie,18854166.0,An unexpected encounter gives a wickedly smart...,United States,0,7.495181,113.0,1
3,MV4,The Father,2021-02-26,United Kingdom|France|United States,Drama|Mystery,8.2,242200.0,Florian Zeller,Anthony Hopkins | Olivia Colman | Mark Gatiss,24048935.0,A man refuses all assistance from his daughter...,United States,0,8.192087,97.0,1
4,MV5,The Invisible Man,2020-02-28,Australia|United States,Drama|Horror|Mystery,7.1,281800.0,Leigh Whannell,Elisabeth Moss | Oliver Jackson-Cohen | Harrie...,144492724.0,When Cecilia's abusive ex takes his own life a...,United States,0,7.097085,124.0,1


In [89]:
# Kiểm tra lại các cột quan trọng
df[['Movie_Title', 'Released_Date', 'IMDB_Rating', 'No_of_votes', 'Weighted_Rating']].head()

,Movie_Title,Released_Date,IMDB_Rating,No_of_votes,Weighted_Rating
0,The Praetorian,2020-02-12,7.6,14.0,6.293892
1,365 Days,2020-06-07,3.3,107900.0,3.327324
2,Promising Young Woman,2020-12-25,7.5,253100.0,7.495181
3,The Father,2021-02-26,8.2,242200.0,8.192087
4,The Invisible Man,2020-02-28,7.1,281800.0,7.097085


In [90]:
# Đảm bảo năm chỉ nằm trong khoảng 2020-2025
df['Released_Date'].dt.year.value_counts().sort_index()

Released_Date
2020     7539
2021     9494
2022    10837
2023    11452
2024    11537
2025    10529
Name: count, dtype: int64

## 17. Lưu file Phim_Clean.csv

In [91]:
df.to_csv("Phim_Clean.csv", index=False, encoding="utf-8-sig")

print("Đã lưu file Phim_Clean.csv thành công!")

Đã lưu file Phim_Clean.csv thành công!


# III. Tách dữ liệu ra thành nhiều sheet trong Excel để vẽ dashboard

## 1. Tạo bản dữ liệu dùng cho dashboard

In [92]:
df_dashboard = df.copy()

## 2. Tạo sheet movies_main

In [93]:
movies_main = df_dashboard[
    [
        'ID',
        'Movie_Title',
        'Released_Date',
        'Release_Country',
        'IMDB_Rating',
        'Weighted_Rating',
        'No_of_votes',
        'Gross',
        'has_gross',
        'Runtime_min',
        'is_imputed_date',
        'Overview'
    ]
].copy()

## 3. Tạo hàm tách dữ liệu nhiều giá trị

In [94]:
def tach_sheet(data, cot_goc, cot_moi):
    temp = data[['ID', cot_goc]].copy()
    temp[cot_goc] = temp[cot_goc].astype(str).str.split('|')
    temp = temp.explode(cot_goc)
    temp[cot_goc] = temp[cot_goc].str.strip()
    temp = temp.rename(columns={cot_goc: cot_moi})
    return temp[['ID', cot_moi]].reset_index(drop=True)

## 4. Tạo các sheet phụ

In [95]:
movies_genre = tach_sheet(df_dashboard, 'Genre', 'Genre_Name')
movies_director = tach_sheet(df_dashboard, 'Director', 'Director_Name')
movies_stars = tach_sheet(df_dashboard, 'Stars', 'Star_Name')
movies_country = tach_sheet(df_dashboard, 'Countries_of_origin', 'Country_of_Origin')

## 5. Lưu file Excel nhiều sheet

In [96]:
output_excel = "DuLieu_Phim_2020_2025.xlsx"

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    movies_main.to_excel(writer, sheet_name='movies_main', index=False)
    movies_genre.to_excel(writer, sheet_name='movies_genre', index=False)
    movies_director.to_excel(writer, sheet_name='movies_director', index=False)
    movies_stars.to_excel(writer, sheet_name='movies_stars', index=False)
    movies_country.to_excel(writer, sheet_name='movies_country', index=False)

print("Đã lưu file Excel nhiều sheet thành công!")

Đã lưu file Excel nhiều sheet thành công!


# IV. Tiền xử lý dữ liệu cho việc chạy mô hình đề xuất phim

## 1. Tạo bản dữ liệu cho hệ thống đề xuất

In [97]:
from pathlib import Path
import re

df_model = df.copy()

## 2. Thiết lập đường dẫn lưu file đầu ra

In [98]:
OUTPUT_DIR = Path(r"D:\Năm3_Kì2\CAP1\HeThongDeXuat")
OUTPUT_CSV = OUTPUT_DIR / "movies_recommender_Co_Quoc_Gia.csv"

## 3. Chuẩn hóa kiểu dữ liệu

In [99]:
# Chuyển ngày phát hành sang kiểu datetime
df_model["Released_Date"] = pd.to_datetime(df_model["Released_Date"], errors="coerce")

# Chuyển các cột số sang numeric
for col in ["IMDB_Rating", "No_of_votes", "Runtime_min", "Weighted_Rating"]:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

## 4. Lọc dữ liệu trong giai đoạn 2020–2025

In [100]:
start_date = pd.Timestamp("2020-01-01")
end_date = pd.Timestamp("2025-12-31")

df_model = df_model[
    (df_model["Released_Date"] >= start_date) &
    (df_model["Released_Date"] <= end_date)
].copy()

print("Kích thước dữ liệu sau khi lọc 2020–2025:", df_model.shape)

Kích thước dữ liệu sau khi lọc 2020–2025: (61388, 16)


## 5. Tạo hàm làm sạch dữ liệu văn bản

In [101]:
def clean_text_for_recommender(x):
    """
    Quy tắc:
    - NaN -> ""
    - Unknown / No description / nan / none / null / n/a -> ""
    - bỏ khoảng trắng thừa
    """
    if pd.isna(x):
        return ""
    
    x = str(x).strip()
    x_lower = x.lower()

    missing_markers = {
        "unknown",
        "no description",
        "nan",
        "none",
        "null",
        "n/a",
        "na"
    }

    if x_lower in missing_markers:
        return ""

    # bỏ khoảng trắng thừa
    x = re.sub(r"\s+", " ", x).strip()

    return x

## 6. Làm sạch các cột văn bản dùng cho mô hình đề xuất

In [102]:
text_cols = ["Genre", "Director", "Stars", "Overview", "Countries_of_origin"]

for col in text_cols:
    if col in df_model.columns:
        df_model[col] = df_model[col].apply(clean_text_for_recommender)

## 7. Loại bỏ các dòng không đủ thông tin cho đề xuất phim

In [103]:
# Bỏ nếu thiếu ID hoặc Movie_Title
df_model = df_model.dropna(subset=["ID", "Movie_Title"]).copy()

# Chuẩn hóa Movie_Title
df_model["Movie_Title"] = df_model["Movie_Title"].astype(str).str.strip()

# Bỏ nếu Movie_Title rỗng
df_model = df_model[df_model["Movie_Title"] != ""].copy()

# Chỉ bỏ khi cả 4 cột nội dung đều rỗng
content_cols = ["Genre", "Director", "Stars", "Overview"]
df_model = df_model[
    ~(df_model[content_cols].eq("").all(axis=1))
].copy()

print("Kích thước dữ liệu sau khi bỏ các dòng không đủ thông tin:", df_model.shape)

Kích thước dữ liệu sau khi bỏ các dòng không đủ thông tin: (61338, 16)


## 8. Tạo bộ dữ liệu cuối cùng cho mô hình đề xuất

In [104]:
# Tạo cột năm phát hành từ Released_Date
df_model["Release_Year"] = df_model["Released_Date"].dt.year.astype("Int64")

final_cols = [
    "ID",
    "Movie_Title",
    "Genre",
    "Director",
    "Stars",
    "Overview",
    "Weighted_Rating",
    "Runtime_min",
    "Release_Year",
    "Countries_of_origin"
]

movies_recommender = df_model[final_cols].copy()
movies_recommender.head()

,ID,Movie_Title,Genre,Director,Stars,Overview,Weighted_Rating,Runtime_min,Release_Year,Countries_of_origin
0,MV1,The Praetorian,Thriller,Nelson Ricardo,Nelson Ricardo | Bello Patricia | Nicholas Apo...,David Donovan was a Bodyguard for hire who wil...,6.293892,93.0,2020,United States
1,MV2,365 Days,Drama|Romance,Barbara Bialowas | Tomasz Mandes,Anna-Maria Sieklucka | Michele Morrone | Broni...,Massimo is a member of the Sicilian Mafia fami...,3.327324,114.0,2020,Poland
2,MV3,Promising Young Woman,Crime|Drama|Mystery,Emerald Fennell,Carey Mulligan | Bo Burnham | Alison Brie,An unexpected encounter gives a wickedly smart...,7.495181,113.0,2020,United Kingdom|United States
3,MV4,The Father,Drama|Mystery,Florian Zeller,Anthony Hopkins | Olivia Colman | Mark Gatiss,A man refuses all assistance from his daughter...,8.192087,97.0,2021,United Kingdom|France|United States
4,MV5,The Invisible Man,Drama|Horror|Mystery,Leigh Whannell,Elisabeth Moss | Oliver Jackson-Cohen | Harrie...,When Cecilia's abusive ex takes his own life a...,7.097085,124.0,2020,Australia|United States


## 9. Kiểm tra bộ dữ liệu đề xuất sau xử lý

In [105]:
print("Kích thước dataset recommender:", movies_recommender.shape)
print("\nSố lượng thiếu theo cột:")
print(movies_recommender.isnull().sum())

Kích thước dataset recommender: (61338, 10)

Số lượng thiếu theo cột:
ID                     0
Movie_Title            0
Genre                  0
Director               0
Stars                  0
Overview               0
Weighted_Rating        0
Runtime_min            0
Release_Year           0
Countries_of_origin    0
dtype: int64


## 10. Lưu file dữ liệu đầu vào cho hệ thống đề xuất

In [106]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
movies_recommender.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Đã lưu xong file: {OUTPUT_CSV}")

Đã lưu xong file: D:\Năm3_Kì2\CAP1\HeThongDeXuat\movies_recommender_Co_Quoc_Gia.csv
